In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raesiow/nova-tf-idf/cm_tfidf.png
/kaggle/input/datasets/raesiow/nova-tf-idf/baseline_preds.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/cm_text_vs_both.png


In [2]:

# put the path from Cell 1 here
PATH = "/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv"
# codes stay text
matched = pd.read_csv(PATH, dtype={"fpx_code": str, "off_code": str})
# expect (1884, 22)
print(matched.shape)
# 50 random pairs, fixed seed
audit = matched.sample(n=50, random_state=42).reset_index(drop=True)
# how many of each matching method ended up in the sample
print(audit["source"].value_counts())

(1884, 22)
source
both         26
mini_only    20
rf_only       4
Name: count, dtype: int64


In [3]:
# go through the 50 sampled pairs
for i, row in audit.iterrows():
    # number, NOVA label, matching method
    print(i, "| NOVA", row["nova"], "|", row["source"])
    # generic food name
    print("   FoodProX:", row["fpx_name"])
    # product name
    print("   OFF:     ", row["off_name"])
    # OFF food group (helps judge)
    print("   group:   ", row["pnns_groups_2"])
    # first 150 characters of ingredients
    print("   ingr:    ", str(row["ingredients_text"])[:150])
    # blank line between pairs
    print()

0 | NOVA 4 | mini_only
   FoodProX: soup beef broth cubed prepared with water
   OFF:      beef broth
   group:    unknown
   ingr:     BOUILLON DE BCEUF (EAU, BOUILLON DE BCEUF), ASSAISONNEMENT (EAU, AROME NATUREL, SEL, CAROTTES, CÉLERI, OIGNON, SUCRE), SEL, DEXTROSE, EXTRAIT DE LEVUR

1 | NOVA 4 | mini_only
   FoodProX: corn chips reduced sodium
   OFF:      lightly salted potato chips
   group:    Appetizers
   ingr:     Potatoes, Vegetable oil, Salt.

2 | NOVA 3 | both
   FoodProX: cereals ready-to-eat weetabix whole grain cereal
   OFF:      whole grain cereal
   group:    Breakfast cereals
   ingr:     Whole grain wheat, Sugars (cane sugar, barley malt extract), Salt, Vitamins and minerals (thiamine mononitrate [vitamin B1], iron, niacinamide [vitami

3 | NOVA 3 | both
   FoodProX: pork bacon smoked or cured cooked
   OFF:      cured bacon
   group:    Processed meat
   ingr:     PORK, WATER, SEA SALT, CULTURED CELERY POWDER

4 | NOVA 3 | both
   FoodProX: mixed nuts roasted with

In [5]:
# your verdicts for pairs 0 to 49, in order; edit only REVIEW lines if you change your mind
verdicts = [
    "right",   # 0
    "right",   # 1  REVIEW
    "right",   # 2
    "right",   # 3
    "right",   # 4
    "unsure",  # 5  REVIEW
    "right",   # 6
    "wrong",   # 7
    "right",   # 8
    "right",   # 9
    "right",   # 10
    "right",   # 11 REVIEW
    "right",   # 12
    "right",   # 13 REVIEW
    "unsure",  # 14 REVIEW
    "wrong",   # 15
    "right",   # 16 REVIEW
    "wrong",   # 17
    "right",   # 18 REVIEW
    "right",   # 19
    "right",   # 20
    "right",   # 21
    "right",   # 22
    "right",   # 23
    "right",   # 24
    "right",   # 25 REVIEW
    "right",   # 26
    "right",   # 27
    "right",   # 28
    "right",   # 29
    "right",   # 30
    "right",   # 31
    "right",   # 32 REVIEW
    "right",   # 33 REVIEW
    "right",   # 34
    "right",   # 35 REVIEW
    "wrong",   # 36
    "right",   # 37
    "right",   # 38
    "right",   # 39
    "wrong",   # 40 REVIEW
    "right",   # 41
    "right",   # 42
    "right",   # 43
    "right",   # 44
    "wrong",   # 45
    "right",   # 46 REVIEW
    "right",   # 47
    "right",   # 48 REVIEW
    "right",   # 49
]
# safety check: must be exactly 50
print("number of verdicts:", len(verdicts))
# attach verdicts to the sampled pairs
audit["verdict"] = verdicts
# how many right / wrong / unsure
print(audit["verdict"].value_counts())
# count the right ones
n_right = (audit["verdict"] == "right").sum()
# main number: unsure counted as wrong
print("precision (unsure = wrong):", round(n_right / 50, 2))
# second number: unsure pairs left out
print("precision (unsure left out):", round(n_right / (audit["verdict"] != "unsure").sum(), 2))
# tool for a 95% confidence interval
from statsmodels.stats.proportion import proportion_confint
# Wilson interval for the main number
low, high = proportion_confint(n_right, 50, method="wilson")
# print it as a range
print("95% CI:", round(low, 2), "-", round(high, 2))
# verdicts split by matching method
print(pd.crosstab(audit["source"], audit["verdict"]))
# save the audit with verdicts
audit.to_csv("/kaggle/working/nova_match_audit.csv", index=False)

number of verdicts: 50
verdict
right     42
wrong      6
unsure     2
Name: count, dtype: int64
precision (unsure = wrong): 0.84
precision (unsure left out): 0.88
95% CI: 0.71 - 0.92
verdict    right  unsure  wrong
source                         
both          23       1      2
mini_only     16       1      3
rf_only        3       0      1


In [7]:
# final verdicts for pairs 0 to 49 (author verdicts; second opinion used where we differed)
verdicts = [
    "right",   # 0
    "wrong",   # 1  changed: corn vs potato chips
    "right",   # 2
    "right",   # 3
    "right",   # 4
    "right",   # 5  changed: sweet dinner rolls = sweet rolls
    "right",   # 6
    "wrong",   # 7
    "right",   # 8
    "right",   # 9
    "right",   # 10
    "wrong",   # 11 changed: raw vs packed segments
    "right",   # 12
    "unsure",  # 13 changed: mayo-type dressing vs real mayo
    "right",   # 14 changed: same fat-free process cheese
    "wrong",   # 15
    "wrong",   # 16 changed: dry mix vs ready soup
    "wrong",   # 17
    "wrong",   # 18 changed: canned vienna vs garlic sausage
    "right",   # 19
    "right",   # 20
    "right",   # 21
    "right",   # 22
    "right",   # 23
    "right",   # 24
    "unsure",  # 25 changed: margarine-like spread vs butter blend
    "right",   # 26
    "right",   # 27
    "right",   # 28
    "right",   # 29
    "right",   # 30
    "right",   # 31
    "wrong",   # 32 changed: with vegetables vs plain
    "wrong",   # 33 changed: plain steak vs seasoned product
    "right",   # 34
    "unsure",  # 35 changed: dry curd vs regular cottage cheese
    "wrong",   # 36
    "right",   # 37
    "right",   # 38
    "right",   # 39
    "right",   # 40 changed: chocolate drink = chocolate soft drink
    "right",   # 41
    "right",   # 42
    "right",   # 43
    "right",   # 44
    "wrong",   # 45
    "wrong",   # 46 changed: tortilla vs pita
    "right",   # 47
    "unsure",  # 48 changed: brown vs probably white rice
    "right",   # 49
]
# safety check: must be exactly 50
print("number of verdicts:", len(verdicts))
# attach verdicts to the sampled pairs
audit["verdict"] = verdicts
# how many right / wrong / unsure
print(audit["verdict"].value_counts())
# count the right ones
n_right = (audit["verdict"] == "right").sum()
# main number: unsure counted as wrong
print("precision (unsure = wrong):", round(n_right / 50, 2))
# second number: unsure pairs left out
print("precision (unsure left out):", round(n_right / (audit["verdict"] != "unsure").sum(), 2))
# tool for a 95% confidence interval
from statsmodels.stats.proportion import proportion_confint
# Wilson interval for the main number
low, high = proportion_confint(n_right, 50, method="wilson")
# print it as a range
print("95% CI:", round(low, 2), "-", round(high, 2))
# verdicts split by matching method
print(pd.crosstab(audit["source"], audit["verdict"]))
# save the audit with verdicts
audit.to_csv("/kaggle/working/nova_match_audit.csv", index=False)

number of verdicts: 50
verdict
right     34
wrong     12
unsure     4
Name: count, dtype: int64
precision (unsure = wrong): 0.68
precision (unsure left out): 0.74
95% CI: 0.54 - 0.79
verdict    right  unsure  wrong
source                         
both          22       2      2
mini_only     10       2      8
rf_only        2       0      2
